<a href="https://colab.research.google.com/github/vinodsri/Applied-Gen-AI/blob/main/ROGUE_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.7 MB/s eta 0:00:00


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
# Import the libraries
import os
import PyPDF2
from evaluate import load
import pandas as pd

In [ ]:
# Define the PDF file path
pdf_path = "arxiv_impact_of_GENAI.pdf"

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"Error: PDF file '{pdf_path}' not found.")

# Read the PDF file
with open(pdf_path, "rb") as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    document_text = ""
    for page in pdf_reader.pages:
        page_text = page.extract_text()
        if page_text:
            document_text += page_text + " "
        else:
            print(f" Warning: Could not extract text from a page.")


In [ ]:
import os
from PyPDF2 import PdfReader

# Define file paths
human_summary_path = "Summary.pdf"
ai_summary_path = "AISUMMARY.pdf"

# Check if summary files exist
if not os.path.exists(human_summary_path):
    raise FileNotFoundError(f"Error: Human summary file '{human_summary_path}' not found.")
if not os.path.exists(ai_summary_path):
    raise FileNotFoundError(f"Error: AI summary file '{ai_summary_path}' not found.")

# Read summaries
def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join([page.extract_text() for page in reader.pages])

human_summary = read_pdf(human_summary_path)
ai_summary = read_pdf(ai_summary_path)

In [ ]:
def preprocess_text(text):
    """Cleans and normalizes text for better ROUGE evaluation."""
    text = text.replace("\n", " ")  # Remove newlines
    text = text.lower().strip()  # Convert to lowercase & remove extra spaces
    return text

# Preprocess all texts
document_text = preprocess_text(document_text)
human_summary = preprocess_text(human_summary)
ai_summary = preprocess_text(ai_summary)

In [ ]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1bd912d7fd4bb224f97b1653219b5e5a1b1ee19d2ce7427f53f6e679c4d561f3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# Load ROUGE metric
metric = load("rouge")

# Compute ROUGE scores for Human Summary
human_scores = metric.compute(predictions=[human_summary], references=[document_text])

# Compute ROUGE scores for AI-Generated Summary
ai_scores = metric.compute(predictions=[ai_summary], references=[document_text])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Convert ROUGE scores to DataFrame
human_scores_df = pd.DataFrame(human_scores, index=["Human Summary"]).T.round(4)
ai_scores_df = pd.DataFrame(ai_scores, index=["AI Summary"]).T.round(4)

# Combine both scores into a single DataFrame
comparison_df = pd.concat([human_scores_df, ai_scores_df], axis=1)

# Display ROUGE score comparison in a table format
print("\n ROUGE Score Comparison")
print(comparison_df)



 ROUGE Score Comparison
           Human Summary  AI Summary
rouge1            0.2948      0.2244
rouge2            0.0820      0.0441
rougeL            0.1859      0.1171
rougeLsum         0.2766      0.2146
